# 🎭 Emotion Detection from Text — End-to-End ML Project

**Goal:** Given a sentence of text, predict which of 6 emotions it expresses:
`joy`, `sadness`, `anger`, `fear`, `love`, `surprise`.

**Dataset:** The `Emotions` dataset (3 files: `train.txt`, `val.txt`, `test.txt`),
each line formatted as `<sentence>;<emotion label>`.

**Pipeline covered in this notebook (matches the assignment steps):**
1. Load the dataset
2. Data cleaning — lowercase, stopword removal, stemming, lemmatization
3. Feature engineering — Bag-of-Words, TF-IDF, Word2Vec (comparison)
4. Train ML models — Logistic Regression, Decision Tree, Random Forest
5. Hyperparameter tuning — GridSearchCV / RandomizedSearchCV
6. Final comparison table + save the best model for the Streamlit app

Every code cell is preceded by a markdown cell explaining **why** that step
exists, not just what the code does — read them in order, they build on
each other.

> This notebook and `train_model.py` (a plain Python script version of the
> same pipeline) produce **identical results** — use the notebook to learn/
> explore interactively, and use `train_model.py` when you just want to
> (re)train and save the model quickly from the command line.


## 0. Setup — Imports and configuration

**Why:** We group all imports at the top so it's immediately clear what
libraries this project depends on (also documented in `requirements.txt`).
We also set a `RANDOM_STATE` constant used everywhere we need randomness
(train/test splits, model initialisation) so that **re-running this
notebook produces the exact same results every time** — critical for
reproducible experiments and fair model comparison.

**Note on file paths:** this notebook lives in the `notebooks/` subfolder,
while `data/`, `models/`, and `utils.py` live one level up in the project
root. The cell below adds the project root to Python's import path (so
`import utils` works) and builds `DATA_DIR`/`MODELS_DIR` relative to this
notebook's own location — this makes the notebook work correctly
regardless of what "current working directory" your Jupyter/VS Code setup
happens to launch from, which is a common source of `FileNotFoundError` /
`ModuleNotFoundError` surprises otherwise.


In [1]:
import os
import sys
import time
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from gensim.models import Word2Vec

# Make the project root (one level above this notebooks/ folder) importable
# and build absolute paths to data/ and models/, so this notebook runs
# correctly no matter which directory Jupyter was launched from.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# utils.py sits in the project root and holds the text-cleaning functions
# shared with app.py (the Streamlit app) — see the file for why that
# sharing matters (training/serving consistency).
from utils import clean_text, load_emotion_file

warnings.filterwarnings("ignore")

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
RANDOM_STATE = 42
MAX_FEATURES = 5000
W2V_DIM = 100

os.makedirs(MODELS_DIR, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Setup complete.")

Project root: /home/claude/emo_proj
Setup complete.


## Step 1 — Load the dataset

**Why this step matters:** Before any modelling can happen we need labelled
examples: a sentence paired with the emotion it expresses. The dataset
ships as three separate files:

- `train.txt` — used to fit (train) the models
- `val.txt` — traditionally used for validation; here we **fold it into the
  training pool** because our hyperparameter search (GridSearchCV /
  RandomizedSearchCV, used later) already performs its own internal
  cross-validation splits, so a separate validation file would be redundant
- `test.txt` — kept **completely untouched** until the very last step, so
  the accuracy we report is an honest estimate of how the model performs on
  sentences it has never seen in any form.

Each line in the raw files looks like:
```
i didnt feel humiliated;sadness
```
i.e. `<sentence>;<emotion label>`. `load_emotion_file()` (in `utils.py`)
parses this format into a tidy pandas DataFrame with `text` and `emotion`
columns.


In [2]:
train_df = load_emotion_file(os.path.join(DATA_DIR, "train.txt"))
val_df = load_emotion_file(os.path.join(DATA_DIR, "val.txt"))
test_df = load_emotion_file(os.path.join(DATA_DIR, "test.txt"))

full_train_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Training pool (train+val): {full_train_df.shape[0]} sentences")
print(f"Held-out test set        : {test_df.shape[0]} sentences")
full_train_df.head()

Training pool (train+val): 18000 sentences
Held-out test set        : 2000 sentences


,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
# Understanding the class balance matters BEFORE modelling: if one emotion
# (e.g. "joy") appears far more often than another (e.g. "surprise"), a lazy
# model could get high accuracy just by always predicting the majority class.
# This is exactly why we'll report *weighted F1-score* later, not just
# accuracy — weighted F1 accounts for this imbalance.
full_train_df["emotion"].value_counts()

emotion
joy         6066
sadness     5216
anger       2434
fear        2149
love        1482
surprise     653
Name: count, dtype: int64

## Step 2 — Data Cleaning

**Why this step matters:** Raw text is noisy from a machine-learning
perspective. `"I am NOT feeling Happy!!"` and `"i am not feeling happy"`
mean almost the same thing to a human, but a naive model would treat every
distinct spelling/casing/punctuation combination as a completely different
feature — exploding the vocabulary size without adding useful signal.
Cleaning fixes this. The specific steps (all implemented in
`utils.clean_text`) are:

| Step | What it does | Why it helps |
|---|---|---|
| **Lowercase** | `"Happy"` → `"happy"` | Prevents the same word being counted as two different features |
| **Remove non-letters** | strips numbers, punctuation, symbols | These add noise; emotion is carried by words, not punctuation, for a classic ML model |
| **Remove stopwords** | drops `"the"`, `"is"`, `"a"`, etc. | These appear in nearly every sentence regardless of emotion — zero discriminative value, pure noise/dimensionality |
| **Stemming** *(alternative)* | crude rule-based chopping, e.g. `"crying"` → `"cri"` | Fast, but output isn't always a real word |
| **Lemmatization** *(what we use)* | dictionary-based root form, e.g. `"crying"` → `"cry"` | Groups inflected forms of a word together so the model learns ONE representation instead of many; output stays a real, readable word |

We use **lemmatization** as our default (see `utils.clean_text(method="lemmatize")`)
because it slightly improves quality on short, informal sentences like this
dataset's, while stemming is available as a drop-in alternative
(`method="stem"`) if you want to compare the two yourself.


In [4]:
t0 = time.time()
full_train_df["clean_text"] = full_train_df["text"].apply(lambda x: clean_text(x, method="lemmatize"))
test_df["clean_text"] = test_df["text"].apply(lambda x: clean_text(x, method="lemmatize"))

full_train_df["tokens"] = full_train_df["clean_text"].apply(lambda x: x.split())
test_df["tokens"] = test_df["clean_text"].apply(lambda x: x.split())

print(f"Cleaning finished in {time.time()-t0:.2f}s")

# Compare a few raw vs cleaned sentences side by side
full_train_df[["text", "clean_text"]].head(5)

Cleaning finished in 5.10s


,text,clean_text
0,i didnt feel humiliated,didnt feel humiliated
1,i can go from feeling so hopeless to so damned...,go feeling hopeless damned hopeful around some...
2,im grabbing a minute to post i feel greedy wrong,im grabbing minute post feel greedy wrong
3,i am ever feeling nostalgic about the fireplac...,ever feeling nostalgic fireplace know still pr...
4,i am feeling grouchy,feeling grouchy


In [5]:
# --- OPTIONAL: see the difference between stemming and lemmatization on the
# same sentence, purely for illustration (not used further in the pipeline).
from utils import clean_text as _ct
sample = "I was crying and feeling extremely surprised by the studies he studied"
print("Original          :", sample)
print("Stemmed            :", _ct(sample, method="stem"))
print("Lemmatized (used)  :", _ct(sample, method="lemmatize"))

Original          : I was crying and feeling extremely surprised by the studies he studied
Stemmed            : cri feel extrem surpris studi studi
Lemmatized (used)  : cry feeling extremely surprised study studied


In [6]:
# ML models work with numbers, not text labels. LabelEncoder maps each
# unique emotion string to an integer, e.g. anger=0, fear=1, joy=2 ...
# We fit it on the training labels only, then reuse (never re-fit) it on the
# test labels — this guarantees the same word always maps to the same
# number in both sets. We also SAVE this encoder later so the Streamlit app
# can turn a model's numeric prediction back into a readable emotion word.
label_encoder = LabelEncoder()
y_train_full = label_encoder.fit_transform(full_train_df["emotion"])
y_test = label_encoder.transform(test_df["emotion"])
print("Emotion classes (index -> label):", list(enumerate(label_encoder.classes_)))

Emotion classes (index -> label): [(0, 'anger'), (1, 'fear'), (2, 'joy'), (3, 'love'), (4, 'sadness'), (5, 'surprise')]


## Step 3 — Feature Engineering / Word Embeddings

**Why compare multiple embeddings instead of picking one:** different text
representations capture different kinds of information, and which one
works best is an empirical question specific to this dataset — not
something to assume in advance.

- **Bag-of-Words (BoW):** counts how many times each word appears in a
  sentence. Simple and fast, but every word is treated as equally
  important, and word order is ignored entirely.
- **TF-IDF (Term Frequency–Inverse Document Frequency):** like BoW, but
  down-weights words that appear in almost every sentence (e.g. common verbs)
  and up-weights words that are rarer and more specific to a particular
  sentence — usually a stronger signal than raw counts.
- **Word2Vec:** learns dense vectors such that semantically similar words
  end up close together in vector space (e.g. `"happy"` and `"joyful"` are
  near each other). This captures *meaning* in a way pure counting cannot,
  but needs enough training data/context to learn good vectors and is
  costlier to compute.

**Our approach:** train a fast Logistic Regression on each of the three
representations using an 80/20 split *of the training pool only* (we never
touch the real test set here — that would leak information into our
embedding choice) and keep whichever wins on validation F1-score for the
rest of the notebook.


In [7]:
X_tr_text, X_val_text, y_tr, y_val = train_test_split(
    full_train_df["clean_text"], y_train_full, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y_train_full
)
tokens_tr, tokens_val = train_test_split(
    full_train_df["tokens"], test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)

embedding_results = []

In [8]:
# --- Bag-of-Words ---
t0 = time.time()
bow_vec = CountVectorizer(max_features=MAX_FEATURES)
Xtr_bow = bow_vec.fit_transform(X_tr_text)
Xval_bow = bow_vec.transform(X_val_text)

clf = LogisticRegression(max_iter=300, n_jobs=-1)
clf.fit(Xtr_bow, y_tr)
f1 = f1_score(y_val, clf.predict(Xval_bow), average="weighted")
embedding_results.append({"Embedding": "Bag-of-Words", "Val F1 (weighted)": f1, "Time (s)": round(time.time()-t0, 2)})
print(f"Bag-of-Words -> validation F1: {f1:.4f}")

Bag-of-Words -> validation F1: 0.8909


In [9]:
# --- TF-IDF ---
# ngram_range=(1,2) means we capture both single words ("happy") AND
# two-word phrases ("not happy") as features, which helps catch simple
# negations that a single-word view would miss.
t0 = time.time()
tfidf_vec = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2))
Xtr_tfidf = tfidf_vec.fit_transform(X_tr_text)
Xval_tfidf = tfidf_vec.transform(X_val_text)

clf = LogisticRegression(max_iter=300, n_jobs=-1)
clf.fit(Xtr_tfidf, y_tr)
f1 = f1_score(y_val, clf.predict(Xval_tfidf), average="weighted")
embedding_results.append({"Embedding": "TF-IDF", "Val F1 (weighted)": f1, "Time (s)": round(time.time()-t0, 2)})
print(f"TF-IDF -> validation F1: {f1:.4f}")

TF-IDF -> validation F1: 0.8816


In [10]:
# --- Word2Vec ---
# Word2Vec produces a vector PER WORD, not per sentence, so to classify a
# whole sentence we average the vectors of all its (known) words into one
# fixed-length "sentence vector". This is a simple, standard pooling
# strategy — more advanced options exist (TF-IDF-weighted average, etc.)
# but plain averaging is a reasonable, fast baseline.
t0 = time.time()
w2v_model = Word2Vec(sentences=tokens_tr.tolist(), vector_size=W2V_DIM, window=5,
                      min_count=2, workers=1, epochs=15, seed=RANDOM_STATE)

def sentence_vector(tokens, model, dim=W2V_DIM):
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

Xtr_w2v = np.array([sentence_vector(t, w2v_model) for t in tokens_tr])
Xval_w2v = np.array([sentence_vector(t, w2v_model) for t in tokens_val])

clf = LogisticRegression(max_iter=300, n_jobs=-1)
clf.fit(Xtr_w2v, y_tr)
f1 = f1_score(y_val, clf.predict(Xval_w2v), average="weighted")
embedding_results.append({"Embedding": "Word2Vec", "Val F1 (weighted)": f1, "Time (s)": round(time.time()-t0, 2)})
print(f"Word2Vec -> validation F1: {f1:.4f}")

Word2Vec -> validation F1: 0.3408


In [11]:
embedding_df = pd.DataFrame(embedding_results).sort_values("Val F1 (weighted)", ascending=False)
print(embedding_df.to_string(index=False))

best_embedding = embedding_df.iloc[0]["Embedding"]
print(f"\n>>> Best embedding: {best_embedding} — used for all models below.")

   Embedding  Val F1 (weighted)  Time (s)
Bag-of-Words           0.890866      0.80
      TF-IDF           0.881620      0.97
    Word2Vec           0.340797      6.48

>>> Best embedding: Bag-of-Words — used for all models below.


In [12]:
# Build the FINAL feature matrices (on the FULL training pool + test set)
# using whichever embedding won above, ready to feed into the models.
if best_embedding == "TF-IDF":
    final_vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2))
elif best_embedding == "Bag-of-Words":
    final_vectorizer = CountVectorizer(max_features=MAX_FEATURES)
else:
    final_vectorizer = None

if final_vectorizer is not None:
    X_train_final = final_vectorizer.fit_transform(full_train_df["clean_text"])
    X_test_final = final_vectorizer.transform(test_df["clean_text"])
else:
    w2v_full = Word2Vec(sentences=full_train_df["tokens"].tolist(), vector_size=W2V_DIM,
                         window=5, min_count=2, workers=1, epochs=15, seed=RANDOM_STATE)
    X_train_final = np.array([sentence_vector(t, w2v_full) for t in full_train_df["tokens"]])
    X_test_final = np.array([sentence_vector(t, w2v_full) for t in test_df["tokens"]])

print("Final training feature matrix shape:", X_train_final.shape)
print("Final test feature matrix shape    :", X_test_final.shape)

Final training feature matrix shape: (18000, 5000)
Final test feature matrix shape    : (2000, 5000)


## Step 4 & 5 — Train & Hyperparameter-Tune the ML Models

**Why tune hyperparameters at all:** a model's default settings are almost
never optimal for a specific dataset. `GridSearchCV` (tries **every**
combination in a grid) and `RandomizedSearchCV` (tries a **random sample**
of combinations — much faster when the full grid would be too expensive)
both use cross-validation internally, meaning each candidate configuration
is scored on several different train/validation splits and averaged — this
protects us from picking hyperparameters that just got lucky on one
particular split.

We evaluate three classic ML algorithms, each with a short explanation of
what its hyperparameters control:

1. **Logistic Regression** — a strong, fast linear baseline for text
   classification.
2. **Decision Tree** — a single tree of if/else rules; easy to interpret
   but prone to overfitting text data's high dimensionality if left
   unconstrained.
3. **Random Forest** — an ensemble of many decision trees whose votes are
   averaged; usually more accurate and more robust than a single tree, at
   the cost of slower training.

A shared helper function `evaluate()` scores every tuned model on the
**same untouched test set**, so the final comparison table (Step 6) is
fair.


In [13]:
results = []

def evaluate(name, model, X_test, y_test, train_time, params):
    preds = model.predict(X_test)
    row = {
        "Model": name,
        "Best Params": str(params),
        "Accuracy": accuracy_score(y_test, preds),
        "Precision (weighted)": precision_score(y_test, preds, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_test, preds, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_test, preds, average="weighted", zero_division=0),
        "Train Time (s)": round(train_time, 2),
    }
    results.append(row)
    print(f"{name}: acc={row['Accuracy']:.4f}  f1={row['F1-score (weighted)']:.4f}  "
          f"params={params}  time={row['Train Time (s)']:.1f}s")
    return row

### 4a. Logistic Regression + tuning

**Hyperparameters searched:**
- `C` — inverse regularisation strength. Smaller `C` = stronger
  regularisation = simpler, more conservative decision boundary (helps
  avoid overfitting); larger `C` fits the training data more tightly.
- `solver` — the optimisation algorithm used internally. We search
  `lbfgs` and `saga`, both of which natively support multiclass problems
  (unlike `liblinear`, which in recent scikit-learn versions requires an
  extra one-vs-rest wrapper for datasets with more than 2 classes, like our
  6 emotions — so we intentionally leave it out).


In [14]:
print("Tuning Logistic Regression ...")
t0 = time.time()
lr_grid = {
    "C": [0.1, 1, 5, 10],
    "solver": ["lbfgs", "saga"],
}
lr_search = GridSearchCV(
    LogisticRegression(max_iter=500, random_state=RANDOM_STATE),
    lr_grid, cv=3, scoring="f1_weighted", n_jobs=-1
)
lr_search.fit(X_train_final, y_train_full)
lr_time = time.time() - t0
evaluate("Logistic Regression (tuned)", lr_search.best_estimator_, X_test_final, y_test, lr_time, lr_search.best_params_)

Tuning Logistic Regression ...


Logistic Regression (tuned): acc=0.8835  f1=0.8835  params={'C': 5, 'solver': 'saga'}  time=44.3s


{'Model': 'Logistic Regression (tuned)',
 'Best Params': "{'C': 5, 'solver': 'saga'}",
 'Accuracy': 0.8835,
 'Precision (weighted)': 0.8836279622858747,
 'Recall (weighted)': 0.8835,
 'F1-score (weighted)': 0.8834523617623521,
 'Train Time (s)': 44.3}

### 4b. Decision Tree + tuning

**Hyperparameters searched:**
- `max_depth` — how deep the tree can grow. Unlimited depth on
  thousands of text features almost always memorises the training data
  (overfitting), so we compare several caps against no cap at all.
- `min_samples_split` / `min_samples_leaf` — the minimum number of samples
  required before a node is allowed to split / to exist as a leaf. Higher
  values force the tree to generalise instead of creating a tiny rule for
  a handful of training examples.

We use `RandomizedSearchCV` here since the full grid (4×3×3=36 combinations)
would take noticeably longer than sampling a representative subset.


In [15]:
print("Tuning Decision Tree ...")
t0 = time.time()
dt_grid = {
    "max_depth": [20, 40, 60, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
dt_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    dt_grid, n_iter=12, cv=3, scoring="f1_weighted", n_jobs=-1, random_state=RANDOM_STATE
)
dt_search.fit(X_train_final, y_train_full)
dt_time = time.time() - t0
evaluate("Decision Tree (tuned)", dt_search.best_estimator_, X_test_final, y_test, dt_time, dt_search.best_params_)

Tuning Decision Tree ...


Decision Tree (tuned): acc=0.8715  f1=0.8721  params={'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': None}  time=14.1s


{'Model': 'Decision Tree (tuned)',
 'Best Params': "{'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': None}",
 'Accuracy': 0.8715,
 'Precision (weighted)': 0.8732007151730066,
 'Recall (weighted)': 0.8715,
 'F1-score (weighted)': 0.8721142525624921,
 'Train Time (s)': 14.1}

### 4c. Random Forest + tuning

**Hyperparameters searched:**
- `n_estimators` — how many trees vote in the ensemble. More trees usually
  means more stable, accurate predictions, but training time grows roughly
  linearly with this number.
- `max_depth` / `min_samples_split` — same overfitting-control role as in
  the Decision Tree above, applied to every tree in the forest.

Random Forest is the slowest model to train here (many trees, each on the
full feature matrix), so we use `RandomizedSearchCV` with a modest
`n_iter` to keep tuning time reasonable — an exhaustive grid search would
cost far more time for only a marginal accuracy gain.


In [16]:
print("Tuning Random Forest ...")
t0 = time.time()
rf_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [20, 40, None],
    "min_samples_split": [2, 5],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    rf_grid, n_iter=6, cv=3, scoring="f1_weighted", n_jobs=1, random_state=RANDOM_STATE
)
rf_search.fit(X_train_final, y_train_full)
rf_time = time.time() - t0
evaluate("Random Forest (tuned)", rf_search.best_estimator_, X_test_final, y_test, rf_time, rf_search.best_params_)

Tuning Random Forest ...


Random Forest (tuned): acc=0.8805  f1=0.8813  params={'n_estimators': 200, 'min_samples_split': 2, 'max_depth': None}  time=135.1s


{'Model': 'Random Forest (tuned)',
 'Best Params': "{'n_estimators': 200, 'min_samples_split': 2, 'max_depth': None}",
 'Accuracy': 0.8805,
 'Precision (weighted)': 0.8825778528867452,
 'Recall (weighted)': 0.8805,
 'F1-score (weighted)': 0.8813412542746478,
 'Train Time (s)': 135.13}

## Step 6 — Final Comparison Table & Saving the Best Model

**Why a single comparison table matters:** we've now tuned three different
models independently. Lining them all up on the *same* held-out test set,
with the *same* metrics, is what makes it possible to pick a genuine winner
rather than comparing apples to oranges.

We use **weighted F1-score** (not plain accuracy) as the primary ranking
metric, because it balances precision and recall *and* accounts for the
class imbalance we saw back in Step 1 (`joy` and `sadness` are far more
common than `surprise`) — a model that's great at the common classes but
terrible at rare ones will be penalised here, whereas accuracy alone could
hide that weakness.

Finally, we **persist** (save to disk) everything the Streamlit app needs
to make predictions instantly, without retraining:
- the winning trained model
- the fitted vectorizer (so new text is converted into the *exact* same
  feature space the model was trained on)
- the label encoder (to translate the model's numeric output back into a
  readable emotion word)


In [17]:
comparison_df = pd.DataFrame(results).sort_values("F1-score (weighted)", ascending=False).reset_index(drop=True)
print(f"FINAL MODEL COMPARISON (embedding used: {best_embedding}):\n")
comparison_df

FINAL MODEL COMPARISON (embedding used: Bag-of-Words):



,Model,Best Params,Accuracy,Precision (weighted),Recall (weighted),F1-score (weighted),Train Time (s)
0,Logistic Regression (tuned),"{'C': 5, 'solver': 'saga'}",0.8835,0.883628,0.8835,0.883452,44.30
1,Random Forest (tuned),"{'n_estimators': 200, 'min_samples_split': 2, ...",0.8805,0.882578,0.8805,0.881341,135.13
2,Decision Tree (tuned),"{'min_samples_split': 10, 'min_samples_leaf': ...",0.8715,0.873201,0.8715,0.872114,14.10


In [18]:
comparison_df.to_csv(os.path.join(MODELS_DIR, "comparison_table.csv"), index=False)
embedding_df.to_csv(os.path.join(MODELS_DIR, "embedding_comparison.csv"), index=False)

best_row = comparison_df.iloc[0]
best_model_name = best_row["Model"]
print(f">>> BEST MODEL: {best_model_name}")
print(f"    F1-weighted = {best_row['F1-score (weighted)']:.4f}")
print(f"    Accuracy    = {best_row['Accuracy']:.4f}")

>>> BEST MODEL: Logistic Regression (tuned)
    F1-weighted = 0.8835
    Accuracy    = 0.8835


In [19]:
model_map = {
    "Logistic Regression (tuned)": lr_search.best_estimator_,
    "Decision Tree (tuned)": dt_search.best_estimator_,
    "Random Forest (tuned)": rf_search.best_estimator_,
}
best_model = model_map[best_model_name]

joblib.dump(best_model, os.path.join(MODELS_DIR, "best_model.pkl"))
joblib.dump(label_encoder, os.path.join(MODELS_DIR, "label_encoder.pkl"))
joblib.dump({"type": best_embedding}, os.path.join(MODELS_DIR, "embedding_info.pkl"))

if final_vectorizer is not None:
    joblib.dump(final_vectorizer, os.path.join(MODELS_DIR, "vectorizer.pkl"))
else:
    w2v_full.save(os.path.join(MODELS_DIR, "word2vec.model"))

print(f"Saved best model + preprocessing artifacts to: {MODELS_DIR}/")
print("You can now launch the web app with:  streamlit run app.py")

Saved best model + preprocessing artifacts to: /home/claude/emo_proj/models/
You can now launch the web app with:  streamlit run app.py


## Next step: try the model in a web app

Everything needed by the Streamlit app (`app.py`) has now been saved into
`models/`. From a terminal, in this project folder, run:

```bash
streamlit run app.py
```

Then open the local URL Streamlit prints (usually `http://localhost:8501`)
and type any sentence to see the predicted emotion, its confidence, and how
your text was cleaned before prediction.

See `README.md` in this project for the complete setup-from-scratch
instructions (virtual environment, installing dependencies, etc.).
